In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import os
from PIL import Image
import numpy as np
from tqdm.auto import tqdm  # これを追加

DATA_DIR = '/mnt/data1/gotou/projects/Medical/kaggledata'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')
LABELS_CSV = os.path.join(DATA_DIR, 'train_labels.csv')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test')

# データ拡張（学習時のみ）
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PCamDataset(Dataset):
    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir = img_dir
        self.labels = labels_df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img_id = self.labels.iloc[idx, 0]
        label = self.labels.iloc[idx, 1]
        img_path = os.path.join(self.img_dir, f"{img_id}.tif")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

labels_df = pd.read_csv(LABELS_CSV)
train_df, val_df = train_test_split(labels_df, test_size=0.1, random_state=42, stratify=labels_df['label'])

train_dataset = PCamDataset(TRAIN_IMG_DIR, train_df, train_transform)
val_dataset = PCamDataset(TRAIN_IMG_DIR, val_df, val_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

resnet50.fc = nn.Linear(resnet50.fc.in_features, 1)
model = resnet50.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# EarlyStopping & モデル保存
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path='best_model_weights.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.best_model = None
        self.path = path

    def __call__(self, val_acc, model):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        '''ベストモデルを保存'''
        torch.save(model.state_dict(), self.path)
        # deepcopy して参照を切る
        self.best_model = copy.deepcopy(model.state_dict())
        if self.verbose:
            print(f'Validation accuracy improved → saving model to {self.path}')


In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# モデル定義（学習時と同じ）
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 1)  # 1ユニット出力
model = model.to(device)

# 重みロード
ckpt_path = "/mnt/data1/gotou/projects/Medical/kaggledata/best_model_weights.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded model from {ckpt_path}")


/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded model from /mnt/data1/gotou/projects/Medical/kaggledata/best_model_weights.pth


In [5]:
import torch
import torch.nn.functional as F

# ===== 設定 (学習時の正規化と一致させる) =====
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
mean = torch.tensor(MEAN).view(1,3,1,1).to(device)
std  = torch.tensor(STD).view(1,3,1,1).to(device)

def denormalize(x, mean, std):
    # normalized -> pixel [0,1]
    return x * std + mean

def renormalize(x_pixel, mean, std):
    # pixel [0,1] -> normalized
    return (x_pixel - mean) / std

def pgd_attack_improved(
    model,
    images,
    labels,
    epsilon_pixel,      # scalar e.g. 8/255 or tensor shape [C]
    alpha_pixel,        # single-step size in pixel scale (e.g. 2/255)
    steps=10,
    device=None,
    mean_tensor=None,
    std_tensor=None,
    random_start=True,
    return_preds=True,
):
    """
    PGD (L_inf) attack that respects per-channel normalization.
    - images: normalized tensor [B,C,H,W]
    - labels: tensor [B] (0/1) or shape [B,1]
    - epsilon_pixel, alpha_pixel: pixel-scale floats (0..1) or 1D tensor per channel
    - steps: number of PGD iterations
    - random_start: initialize inside L_inf-ball uniformly
    Returns:
      adv_images (normalized, detached) and adv_preds (cpu LongTensor) if return_preds True
    """
    # use provided mean/std or globals
    if mean_tensor is None or std_tensor is None:
        mean_tensor_local = mean
        std_tensor_local = std
    else:
        mean_tensor_local = mean_tensor
        std_tensor_local = std_tensor

    if device is None:
        device = images.device

    images = images.clone().detach().to(device)
    labels = labels.clone().detach().to(device)

    B, C, H, W = images.shape

    # convert eps and alpha to tensors on device and pixel space shape [1,C,1,1]
    def _to_channel_tensor(x):
        if not torch.is_tensor(x):
            t = torch.tensor(x, dtype=images.dtype, device=device)
        else:
            t = x.to(device).to(images.dtype)
        if t.ndim == 0:
            t = t.view(1, 1, 1, 1)  # scalar -> broadcastable
        elif t.ndim == 1 and t.numel() == C:
            t = t.view(1, C, 1, 1)
        else:
            # assume broadcastable already
            t = t.view(1, -1, 1, 1)
        return t

    eps_pixel_t = _to_channel_tensor(epsilon_pixel)   # pixel scale
    alpha_pixel_t = _to_channel_tensor(alpha_pixel)   # pixel scale

    # convert to normalized-space epsilon/alpha
    eps_norm = eps_pixel_t / std_tensor_local.to(images.dtype)
    alpha_norm = alpha_pixel_t / std_tensor_local.to(images.dtype)

    # get pixel-space original images
    orig_pixel = denormalize(images, mean_tensor_local, std_tensor_local)  # [B,C,H,W], pixel in [0,1]

    # random start (in pixel space) then renormalize
    if random_start:
        # uniform perturbation in [-eps, eps] per channel
        uni = torch.empty_like(orig_pixel).uniform_(-1.0, 1.0)
        # scale to [-eps, eps] per channel
        rand_pert = uni * eps_pixel_t
        adv_pixel = torch.clamp(orig_pixel + rand_pert, 0.0, 1.0)
        adv_images = renormalize(adv_pixel, mean_tensor_local, std_tensor_local).detach()
    else:
        adv_images = images.clone().detach()

    # main loop
    adv_images = adv_images.to(device)
    for _ in range(steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        if outputs.ndim > 1 and outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        loss = F.binary_cross_entropy_with_logits(outputs, labels.float())
        model.zero_grad()
        loss.backward()
        grad = adv_images.grad.data

        # sign update in normalized space
        adv_images = adv_images + alpha_norm * grad.sign()

        # project back to L_inf ball around original pixel image:
        # compute pixel space, clamp difference to [-eps_pixel, eps_pixel]
        adv_pixel = denormalize(adv_images, mean_tensor_local, std_tensor_local)
        eta = torch.clamp(adv_pixel - orig_pixel, min=-eps_pixel_t, max=eps_pixel_t)
        adv_pixel = torch.clamp(orig_pixel + eta, 0.0, 1.0)

        # back to normalized space and detach for next iter
        adv_images = renormalize(adv_pixel, mean_tensor_local, std_tensor_local).detach()

        # cleanup grad
        adv_images.requires_grad = False
        del grad, outputs, loss, eta, adv_pixel
        torch.cuda.empty_cache()

    # return preds if requested
    if return_preds:
        with torch.no_grad():
            adv_out = model(adv_images)
            if adv_out.ndim > 1 and adv_out.shape[1] == 1:
                adv_out = adv_out.squeeze(1)
            adv_probs = torch.sigmoid(adv_out)
            adv_preds = (adv_probs > 0.5).long().cpu()
        return adv_images, adv_preds

    return adv_images


# --- 評価ループ（PGD版） ---
def evaluate_clean_and_pgd(
    model, val_loader, device,
    epsilon_pixel=8/255, alpha_pixel=2/255, steps=10,
    mean=mean, std=std,
    attack_only_correct=False, random_start=True,
    max_samples=100
):
    """
    attack_only_correct: True → 正しく分類されたサンプルのみ攻撃
    returns: orig_acc, adv_acc, avg_L2, avg_Linf
    """
    adv_correct, adv_total = 0, 0
    orig_correct, orig_total = 0, 0
    l2_norms, linf_norms = [], []

    model.eval()
    processed = 0

    pbar = tqdm(val_loader, desc=f"PGD attack ({steps} steps)", ncols=100)
    for images, labels in pbar:
        if processed >= max_samples:
            break
        images, labels = images.to(device), labels.to(device)

        # --- 元画像の精度計測 ---
        outputs = model(images)
        if outputs.ndim > 1 and outputs.shape[1] == 1:
            outputs = outputs.squeeze(1)
        preds = (torch.sigmoid(outputs) > 0.5).long()

        if attack_only_correct:
            correct_mask = (preds == labels.long())
            if correct_mask.sum() == 0:
                continue
            images_to_attack = images[correct_mask]
            labels_to_attack = labels[correct_mask]
            orig_correct += correct_mask.sum().item()
            orig_total += correct_mask.sum().item()
        else:
            images_to_attack = images
            labels_to_attack = labels
            orig_correct += (preds == labels.long()).sum().item()
            orig_total += labels.size(0)

        # --- PGD攻撃 ---
        adv_images, adv_preds = pgd_attack_improved(
            model,
            images_to_attack,
            labels_to_attack,
            epsilon_pixel=epsilon_pixel,
            alpha_pixel=alpha_pixel,
            steps=steps,
            device=device,
            mean_tensor=mean,
            std_tensor=std,
            random_start=random_start,
            return_preds=True
        )

        # --- L2, L∞ノルム計算 ---
        orig_pixel = denormalize(images_to_attack, mean, std)
        adv_pixel = denormalize(adv_images, mean, std)
        diff = (adv_pixel - orig_pixel).view(adv_pixel.size(0), -1)

        l2 = diff.norm(p=2, dim=1)        # 各画像のL2ノルム
        linf = diff.abs().max(dim=1)[0]   # 各画像のL∞ノルム

        l2_norms.extend(l2.cpu().numpy().tolist())
        linf_norms.extend(linf.cpu().numpy().tolist())

        adv_correct += (adv_preds == labels_to_attack.cpu()).sum().item()
        adv_total += labels_to_attack.size(0)
        processed += images_to_attack.size(0)

    orig_acc = orig_correct / orig_total * 100 if orig_total > 0 else 0.0
    adv_acc = adv_correct / adv_total * 100 if adv_total > 0 else 0.0
    avg_l2 = np.mean(l2_norms) if len(l2_norms) > 0 else 0.0
    avg_linf = np.mean(linf_norms) if len(linf_norms) > 0 else 0.0

    print(f"Original Accuracy (evaluated subset): {orig_acc:.2f}% ({orig_correct}/{orig_total})")
    print(f"PGD Adversarial Accuracy: {adv_acc:.2f}% ({adv_correct}/{adv_total})")
    print(f"Average L2 norm: {avg_l2:.6f}")
    print(f"Average L∞ norm: {avg_linf:.6f}")

    return orig_acc, adv_acc, avg_l2, avg_linf

In [6]:
orig_acc, adv_acc, avg_l2 = evaluate_clean_and_pgd(
    model, 
    val_loader, 
    device,
    epsilon_pixel=8/255,    # 攻撃強度（一般的な設定）
    alpha_pixel=2/255,      # ステップごとの更新量
    steps=10,               # 繰り返し回数
    mean=mean, 
    std=std,
    attack_only_correct=True,  # FGSMと同条件に揃えるのが比較的おすすめ
    random_start=True,         # ランダム初期化でより強力な攻撃
    max_samples=100            # 評価するサンプル数を制限（重い場合は小さく）
)


PGD attack (10 steps):   0%|                                                | 0/688 [00:00<?, ?it/s]

Original Accuracy (evaluated subset): 100.00% (126/126)
PGD Adversarial Accuracy: 2.38% (3/126)
Average L2 norm: 7.842049
Average L∞ norm: 0.031373


ValueError: too many values to unpack (expected 3)